# Pharma Compliance Spend Analytics — PySpark Implementation

**Third implementation** of the HCP feature-engineering pipeline, after Pandas ([`01_data_features.ipynb`](./01_data_features.ipynb)) and Snowflake SQL ([`../sql/snowflake_queries.sql`](../sql/snowflake_queries.sql)).

**Why a third implementation?** Each platform fits a different scale:

| Platform | Best for | Trade-off |
|---|---|---|
| **Pandas** | Sample-size datasets that fit in RAM | Limited by single-machine memory |
| **Snowflake** | Cloud SQL analytics, multi-user warehouses | Vendor-locked, $ per compute-second |
| **PySpark** | Distributed ETL, multi-TB pipelines | More setup overhead; lazy evaluation requires mental model shift |

**What this notebook does:**
1. Spins up a local Spark session
2. Loads the 989K-row CMS Open Payments sample as a Spark DataFrame
3. Reproduces the 5 HCP-level features using the DataFrame API
4. Reproduces the same features using Spark SQL (parity demo — same answer two ways)
5. Demonstrates Spark-specific concepts: lazy evaluation, query plans, partitioning, caching
6. Saves output and verifies row-level equivalence with the Pandas pipeline

**Author:** Shrikant Sharma — Data Scientist (Pharma & Financial Services)

In [1]:
import os
import sys

# Force PySpark to use the venv's Python, bypassing the Windows Store alias
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# Defensive: also set JAVA_HOME here in case the shell env didn't propagate
os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.19.10-hotspot"

print(f"PYSPARK_PYTHON:        {os.environ['PYSPARK_PYTHON']}")
print(f"PYSPARK_DRIVER_PYTHON: {os.environ['PYSPARK_DRIVER_PYTHON']}")
print(f"JAVA_HOME:             {os.environ['JAVA_HOME']}")

PYSPARK_PYTHON:        c:\Users\shrik\Documents\Projects\pharma-compliance-spend-analytics\.venv\Scripts\python.exe
PYSPARK_DRIVER_PYTHON: c:\Users\shrik\Documents\Projects\pharma-compliance-spend-analytics\.venv\Scripts\python.exe
JAVA_HOME:             C:\Program Files\Eclipse Adoptium\jdk-17.0.19.10-hotspot


In [2]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType, DateType

print(f"PySpark version: {pyspark.__version__}")

PySpark version: 4.1.1


In [3]:
spark = (
    SparkSession.builder
    .appName("PharmaCompliancePipeline")
    .master("local[*]")                              # use all available cores
    .config("spark.driver.memory", "4g")             # plenty for 989K rows
    .config("spark.sql.shuffle.partitions", "8")     # local mode default 200 is overkill
    .config("spark.sql.adaptive.enabled", "true")    # AQE optimizes joins/shuffles at runtime
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")   # suppress chatty INFO logs

print(f"Spark UI: {spark.sparkContext.uiWebUrl}")
print(f"Spark version: {spark.version}")
print(f"Cores allocated: {spark.sparkContext.defaultParallelism}")

Spark UI: http://host.docker.internal:4040
Spark version: 4.1.1
Cores allocated: 16


## HCP Feature Engineering — DataFrame API

Reproduce the 5 HCP-level features from notebook 01 using the PySpark DataFrame API. The transformations below build a *query plan*; no data moves until we trigger an action.

| Feature | How we compute it (PySpark) |
|---|---|
| `total_payment_value` | `F.sum(amount)` per NPI |
| `payment_frequency` | `F.count(*)` per NPI |
| `avg_payment_size` | total / frequency (derived) |
| `n_unique_states` | `F.countDistinct(state)` per NPI |
| `top_company_share` | Two-step: aggregate by (NPI, manufacturer) → take max per NPI → divide by total |

We also compute `primary_state` (most-frequent state per NPI) as a secondary aggregation, using `F.max_by()` to keep the value associated with the highest count.

In [5]:
DATA_PATH = "../output/payments_sampled.csv"

# Read CSV with header detection and schema inference
df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("multiLine", "true")
    .option("escape", '"')
    .csv(DATA_PATH)
)

# Sanity: row count + schema + 5 rows
print(f"Row count: {df.count():,}")
print(f"\nSchema:")
df.printSchema()
print(f"\nFirst 5 rows:")
df.show(5, truncate=50)

Row count: 988,821

Schema:
root
 |-- Covered_Recipient_Type: string (nullable = true)
 |-- Covered_Recipient_NPI: double (nullable = true)
 |-- Covered_Recipient_First_Name: string (nullable = true)
 |-- Covered_Recipient_Last_Name: string (nullable = true)
 |-- Recipient_State: string (nullable = true)
 |-- Covered_Recipient_Specialty_1: string (nullable = true)
 |-- Applicable_Manufacturer_or_Applicable_GPO_Making_Payment_Name: string (nullable = true)
 |-- Total_Amount_of_Payment_USDollars: double (nullable = true)
 |-- Date_of_Payment: string (nullable = true)
 |-- Nature_of_Payment_or_Transfer_of_Value: string (nullable = true)


First 5 rows:
+---------------------------+---------------------+----------------------------+---------------------------+---------------+--------------------------------------------------+-------------------------------------------------------------+---------------------------------+---------------+--------------------------------------+
|     Covered_R

In [6]:
# === Base aggregates ===
hcp_base = (
    df.groupBy("Covered_Recipient_NPI")
    .agg(
        F.first("Covered_Recipient_First_Name").alias("first_name"),
        F.first("Covered_Recipient_Last_Name").alias("last_name"),
        F.first("Covered_Recipient_Specialty_1").alias("specialty"),
        F.sum("Total_Amount_of_Payment_USDollars").alias("total_payment_value"),
        F.count("*").alias("payment_frequency"),
        F.countDistinct("Recipient_State").alias("n_unique_states"),
    )
)

# === Top company per NPI (two-step) ===
# Step 1: aggregate at (NPI, manufacturer) level
company_totals = (
    df.groupBy(
        "Covered_Recipient_NPI",
        "Applicable_Manufacturer_or_Applicable_GPO_Making_Payment_Name",
    )
    .agg(F.sum("Total_Amount_of_Payment_USDollars").alias("company_total"))
)

# Step 2: keep the row with the max company_total per NPI
# F.max_by returns the value of arg1 at the row where arg2 is maximum
top_company = (
    company_totals.groupBy("Covered_Recipient_NPI")
    .agg(
        F.max("company_total").alias("top_company_total"),
        F.max_by(
            "Applicable_Manufacturer_or_Applicable_GPO_Making_Payment_Name",
            "company_total",
        ).alias("top_company_name"),
    )
)

# === Primary state (most-frequent state per NPI) ===
state_counts = (
    df.groupBy("Covered_Recipient_NPI", "Recipient_State")
    .agg(F.count("*").alias("state_count"))
)
primary_state = (
    state_counts.groupBy("Covered_Recipient_NPI")
    .agg(F.max_by("Recipient_State", "state_count").alias("primary_state"))
)

# === Join everything, add derived features, reorder columns ===
hcp_features_dfapi = (
    hcp_base
    .join(primary_state, on="Covered_Recipient_NPI", how="left")
    .join(top_company,   on="Covered_Recipient_NPI", how="left")
    .withColumn("avg_payment_size",
                F.col("total_payment_value") / F.col("payment_frequency"))
    .withColumn("top_company_share",
                F.col("top_company_total") / F.col("total_payment_value"))
    .select(
        "Covered_Recipient_NPI",
        "first_name", "last_name", "specialty", "primary_state",
        "total_payment_value", "payment_frequency", "avg_payment_size",
        "n_unique_states", "top_company_share",
        "top_company_name", "top_company_total",
    )
)

print("Query plan defined. Nothing executed yet — these are lazy transformations.")
print(f"\nColumns: {hcp_features_dfapi.columns}")

Query plan defined. Nothing executed yet — these are lazy transformations.

Columns: ['Covered_Recipient_NPI', 'first_name', 'last_name', 'specialty', 'primary_state', 'total_payment_value', 'payment_frequency', 'avg_payment_size', 'n_unique_states', 'top_company_share', 'top_company_name', 'top_company_total']


In [7]:
import time

t0 = time.time()
row_count_dfapi = hcp_features_dfapi.count()
elapsed = time.time() - t0

print(f"HCP row count (DataFrame API): {row_count_dfapi:,}")
print(f"Time: {elapsed:.1f}s\n")
print("First 5 rows:")
hcp_features_dfapi.show(5, truncate=40)

HCP row count (DataFrame API): 288,942
Time: 2.3s

First 5 rows:
+---------------------+----------+----------+----------------------------------------+-------------+-------------------+-----------------+------------------+---------------+-------------------+-------------------+-----------------+
|Covered_Recipient_NPI|first_name| last_name|                               specialty|primary_state|total_payment_value|payment_frequency|  avg_payment_size|n_unique_states|  top_company_share|   top_company_name|top_company_total|
+---------------------+----------+----------+----------------------------------------+-------------+-------------------+-----------------+------------------+---------------+-------------------+-------------------+-----------------+
|        1.003000902E9|  JAIVANTI|    LOHANO|Allopathic & Osteopathic Physicians|F...|           KY|             144.26|                8|           18.0325|              2|0.24178566477193958|        ABBVIE INC.|            34.88|
|      

## Spark SQL Parity — Same Logic, Different Syntax

The DataFrame API version above can be expressed equivalently in standard SQL with Window functions. This demonstrates that PySpark gives you both interfaces — choose by team familiarity, not capability difference. The same query plan is generated by both, so performance is identical.

Below: register the source DataFrame as a temporary view, then run a single SQL query with CTEs and `ROW_NUMBER() OVER` window functions to get the same HCP-level features.

In [8]:
df.createOrReplaceTempView("payments")

hcp_features_sql = spark.sql("""
WITH base AS (
    SELECT
        Covered_Recipient_NPI AS npi,
        FIRST(Covered_Recipient_First_Name)  AS first_name,
        FIRST(Covered_Recipient_Last_Name)   AS last_name,
        FIRST(Covered_Recipient_Specialty_1) AS specialty,
        SUM(Total_Amount_of_Payment_USDollars) AS total_payment_value,
        COUNT(*)                               AS payment_frequency,
        COUNT(DISTINCT Recipient_State)        AS n_unique_states
    FROM payments
    GROUP BY Covered_Recipient_NPI
),
company_totals AS (
    SELECT
        Covered_Recipient_NPI AS npi,
        Applicable_Manufacturer_or_Applicable_GPO_Making_Payment_Name AS company,
        SUM(Total_Amount_of_Payment_USDollars) AS company_total
    FROM payments
    GROUP BY Covered_Recipient_NPI,
             Applicable_Manufacturer_or_Applicable_GPO_Making_Payment_Name
),
top_company AS (
    SELECT npi, company AS top_company_name, company_total AS top_company_total
    FROM (
        SELECT npi, company, company_total,
               ROW_NUMBER() OVER (PARTITION BY npi ORDER BY company_total DESC) AS rn
        FROM company_totals
    ) WHERE rn = 1
),
state_counts AS (
    SELECT
        Covered_Recipient_NPI AS npi,
        Recipient_State AS state,
        COUNT(*) AS state_count
    FROM payments
    GROUP BY Covered_Recipient_NPI, Recipient_State
),
primary_state AS (
    SELECT npi, state AS primary_state
    FROM (
        SELECT npi, state, state_count,
               ROW_NUMBER() OVER (PARTITION BY npi ORDER BY state_count DESC) AS rn
        FROM state_counts
    ) WHERE rn = 1
)
SELECT
    b.npi                                                          AS Covered_Recipient_NPI,
    b.first_name,
    b.last_name,
    b.specialty,
    ps.primary_state,
    b.total_payment_value,
    b.payment_frequency,
    b.total_payment_value / b.payment_frequency                    AS avg_payment_size,
    b.n_unique_states,
    tc.top_company_total / b.total_payment_value                   AS top_company_share,
    tc.top_company_name,
    tc.top_company_total
FROM base b
LEFT JOIN primary_state ps ON b.npi = ps.npi
LEFT JOIN top_company  tc ON b.npi = tc.npi
""")

t0 = time.time()
row_count_sql = hcp_features_sql.count()
elapsed = time.time() - t0

print(f"HCP row count (Spark SQL):     {row_count_sql:,}")
print(f"DataFrame API count was:       {row_count_dfapi:,}")
print(f"Match: {row_count_sql == row_count_dfapi}")
print(f"Time: {elapsed:.1f}s\n")
print("First 5 rows from SQL version:")
hcp_features_sql.show(5, truncate=40)

HCP row count (Spark SQL):     288,942
DataFrame API count was:       288,942
Match: True
Time: 5.1s

First 5 rows from SQL version:
+---------------------+----------+----------+----------------------------------------+-------------+-------------------+-----------------+------------------+---------------+-------------------+-------------------+-----------------+
|Covered_Recipient_NPI|first_name| last_name|                               specialty|primary_state|total_payment_value|payment_frequency|  avg_payment_size|n_unique_states|  top_company_share|   top_company_name|top_company_total|
+---------------------+----------+----------+----------------------------------------+-------------+-------------------+-----------------+------------------+---------------+-------------------+-------------------+-----------------+
|        1.003000902E9|  JAIVANTI|    LOHANO|Allopathic & Osteopathic Physicians|F...|           KY|             144.26|                8|           18.0325|              

## Spark-Specific Concepts

Three things PySpark gives you that Pandas doesn't: query optimization (visible via `.explain()`), data partitioning across cores, and in-memory caching for repeated reads.

In [9]:
print("Physical query plan for the PySpark HCP DataFrame:")
print("=" * 70)
hcp_features_dfapi.explain(mode="formatted")

Physical query plan for the PySpark HCP DataFrame:
== Physical Plan ==
AdaptiveSparkPlan (41)
+- Project (40)
   +- SortMergeJoin LeftOuter (39)
      :- Project (26)
      :  +- SortMergeJoin LeftOuter (25)
      :     :- Sort (12)
      :     :  +- Exchange (11)
      :     :     +- SortAggregate (10)
      :     :        +- Sort (9)
      :     :           +- Exchange (8)
      :     :              +- SortAggregate (7)
      :     :                 +- SortAggregate (6)
      :     :                    +- Sort (5)
      :     :                       +- Exchange (4)
      :     :                          +- SortAggregate (3)
      :     :                             +- Sort (2)
      :     :                                +- Scan csv  (1)
      :     +- Sort (24)
      :        +- Exchange (23)
      :           +- SortAggregate (22)
      :              +- Sort (21)
      :                 +- Exchange (20)
      :                    +- SortAggregate (19)
      :                      

In [10]:
import time

# Partition awareness
print(f"Source DataFrame partitions: {df.rdd.getNumPartitions()}")
print(f"Result DataFrame partitions: {hcp_features_dfapi.rdd.getNumPartitions()}")
print()

# Caching demo — first count vs second count after cache
print("Caching demo — first count vs second count after cache:")
print("=" * 70)

hcp_features_dfapi.unpersist()  # reset any prior cache state

# Uncached count
t0 = time.time()
n1 = hcp_features_dfapi.count()
t_uncached = time.time() - t0

# Cache, materialize, re-count
hcp_features_dfapi.cache()
hcp_features_dfapi.count()  # this materializes the cache
t0 = time.time()
n2 = hcp_features_dfapi.count()
t_cached = time.time() - t0

print(f"Uncached: {n1:,} rows in {t_uncached:.2f}s")
print(f"Cached:   {n2:,} rows in {t_cached:.3f}s")
print(f"Speedup:  {t_uncached / t_cached:.0f}x")

Source DataFrame partitions: 1
Result DataFrame partitions: 8

Caching demo — first count vs second count after cache:
Uncached: 288,942 rows in 1.60s
Cached:   288,942 rows in 0.225s
Speedup:  7x


## Save + Verify Equivalence with Pandas Pipeline

Save the PySpark-computed HCP features to CSV, then load the Pandas-computed equivalent from notebook 01 and verify they produce the same numbers HCP-by-HCP. This is the credibility step — "I ran the same logic on three platforms and got the same answer" is a stronger claim with this comparison than without.

In [11]:
import os

os.makedirs("../output", exist_ok=True)

# Convert to Pandas and write as single CSV
# (288K rows fits comfortably in memory; this gives a clean single-file output
# rather than Spark's default multi-part directory)
output_pd = hcp_features_dfapi.toPandas()
print(f"Converted to Pandas: {output_pd.shape}")

output_path = "../output/hcp_features_pyspark.csv"
output_pd.to_csv(output_path, index=False)
size_mb = os.path.getsize(output_path) / 1024 / 1024
print(f"Saved: {output_path}")
print(f"Size:  {size_mb:.1f} MB")

Converted to Pandas: (288942, 12)
Saved: ../output/hcp_features_pyspark.csv
Size:  42.5 MB


In [12]:
import pandas as pd
import numpy as np

# Load the Pandas pipeline's output (from notebook 01)
pd_features = pd.read_csv("../output/hcp_features_with_flags.csv")

# PySpark output already in memory as output_pd
spark_features = output_pd

print(f"Pandas pipeline output:  {pd_features.shape}")
print(f"PySpark pipeline output: {spark_features.shape}")
print(f"Row count match: {len(pd_features) == len(spark_features)}")
print()

# Merge on NPI
merged = pd_features.merge(
    spark_features,
    on="Covered_Recipient_NPI",
    suffixes=("_pd", "_spark"),
)
print(f"Merged on NPI: {len(merged):,} rows")
print(f"NPIs in Pandas only:  {len(pd_features) - len(merged):,}")
print(f"NPIs in PySpark only: {len(spark_features) - len(merged):,}")
print()

# Numeric feature equivalence
print("Per-HCP numeric feature equivalence:")
print("=" * 70)
for col in ['total_payment_value', 'payment_frequency', 'n_unique_states']:
    pd_vals = merged[f"{col}_pd"]
    spark_vals = merged[f"{col}_spark"]
    exact_match = (pd_vals == spark_vals).mean() * 100
    print(f"  {col:<25} exact match: {exact_match:.2f}%")

# Floating-point columns: allow 1e-4 tolerance
for col in ['avg_payment_size', 'top_company_share']:
    pd_vals = merged[f"{col}_pd"]
    spark_vals = merged[f"{col}_spark"]
    close_match = np.isclose(pd_vals, spark_vals, rtol=1e-4, atol=1e-4).mean() * 100
    print(f"  {col:<25} match @ 1e-4 tolerance: {close_match:.2f}%")

# Top 5 highest-paid HCPs from each pipeline
print("\nTop 5 highest-paid HCPs — Pandas pipeline:")
top_pd = pd_features.nlargest(5, 'total_payment_value')[
    ['Covered_Recipient_NPI', 'specialty', 'total_payment_value', 'top_company_share']
]
print(top_pd.to_string(index=False))

print("\nTop 5 highest-paid HCPs — PySpark pipeline:")
top_spark = spark_features.nlargest(5, 'total_payment_value')[
    ['Covered_Recipient_NPI', 'specialty', 'total_payment_value', 'top_company_share']
]
print(top_spark.to_string(index=False))

Pandas pipeline output:  (288942, 25)
PySpark pipeline output: (288942, 12)
Row count match: True

Merged on NPI: 288,942 rows
NPIs in Pandas only:  0
NPIs in PySpark only: 0

Per-HCP numeric feature equivalence:
  total_payment_value       exact match: 86.25%
  payment_frequency         exact match: 100.00%
  n_unique_states           exact match: 100.00%
  avg_payment_size          match @ 1e-4 tolerance: 100.00%
  top_company_share         match @ 1e-4 tolerance: 100.00%

Top 5 highest-paid HCPs — Pandas pipeline:
 Covered_Recipient_NPI                                                                                        specialty  total_payment_value  top_company_share
          1669457354.0                          Allopathic & Osteopathic Physicians|Orthopaedic Surgery|Sports Medicine           3571994.60           1.000000
          1215948104.0 Allopathic & Osteopathic Physicians|Orthopaedic Surgery|Adult Reconstructive Orthopaedic Surgery           3200055.07           0.9998